# Лабораторная работа №4
## Хеширование
## Ревенко Данила, ФИТ-231
### № 1
Возьмите реализацию класса HashTable из лекционных материалов и выполните
следующие доработки:
1. Реализуйте квадратичное пробирование как технику повторного хеширования.
2. Реализуйте работу с функцией len (переопределите метод __len__).
3. Реализуйте работу оператора in (переопределите метод __contains__).
4. Переделайте метод put таким образом, чтобы таблица автоматически меняла размер,
когда загрузочный фактор становится больше значения 0.7. Размер должен
увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте работу оператора del (переопределите метод __delitem__) для удаления
элемента таблицы. Таблица должна автоматически менять размер, когда
загрузочный фактор становится меньше значения 0.2. Размер должен уменьшаться
примерно в два раза до ближайшего подходящего простого числа.
Все выполненные доработки должны быть протестированы

In [9]:
class HashTable:
    def __init__(self):
        self.size = 11
        self.slots = [None] * self.size
        self.data = [None] * self.size

    def put(self, key, data):
        hashvalue = self.hashfunction(key, len(self.slots))

        if self.slots[hashvalue] == None:
            self.slots[hashvalue] = key
            self.data[hashvalue] = data
        else:
            if self.slots[hashvalue] == key:
                self.data[hashvalue] = data  # replace
            else:
                nextslot = self.rehash(hashvalue, len(self.slots))
                while self.slots[nextslot] != None and \
                        self.slots[nextslot] != key:
                    nextslot = self.rehash(nextslot, len(self.slots))

                if self.slots[nextslot] == None:
                    self.slots[nextslot] = key
                    self.data[nextslot] = data
                else:
                    self.data[nextslot] = data  # replace

    def hashfunction(self, key, size):
        return key % size

    def rehash(self, oldhash, size):
        return (oldhash + 1) % size

    def get(self, key):
        startslot = self.hashfunction(key, len(self.slots))

        data = None
        stop = False
        found = False
        position = startslot
        while self.slots[position] != None and \
                not found and not stop:
            if self.slots[position] == key:
                found = True
                data = self.data[position]
            else:
                position = self.rehash(position, len(self.slots))
                if position == startslot:
                    stop = True
        return data

    def __getitem__(self, key):
        return self.get(key)

    def __setitem__(self, key, data):
        self.put(key, data)


    # 1) Задание 1: квадратичное пробирование
    def rehash(self, starthash, i, size):
        # квадратичное смещение (start + i^2) % size
        return (starthash + i * i) % size

    def put(self, key, value):
        # реализация вставки с квадратичным пробированием (без resize, без tombstone)
        start = self.hashfunction(key, self.size)
        index = start
        i = 0
        while True:
            slot = self.slots[index]
            if slot is None:
                self.slots[index] = key
                self.data[index] = value
                return
            elif slot == key:
                self.data[index] = value
                return
            i += 1
            if i >= self.size:
                raise RuntimeError("HashTable is full; rehash failed.")
            index = self.rehash(start, i, self.size)

    def get(self, key):
        # поиск с квадратичным пробированием
        start = self.hashfunction(key, self.size)
        index = start
        i = 0
        while True:
            slot = self.slots[index]
            if slot is None:
                return None
            if slot == key:
                return self.data[index]
            i += 1
            if i >= self.size:
                return None
            index = self.rehash(start, i, self.size)

    # 2) Задание 2: поддержка len()
    def __len__(self):
        # количество реально хранимых элементов (будет корректно поддерживаться ниже)
        return getattr(self, "count", 0)

    def __init__(self, initial_size=11):
        # добавлен счетчик элементов
        self.size = 11
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0

    def put(self, key, value):
        # корректная поддержка count при вставке/обновлении (пока без tombstone)
        start = self.hashfunction(key, self.size)
        index = start
        i = 0
        while True:
            slot = self.slots[index]
            if slot is None:
                self.slots[index] = key
                self.data[index] = value
                self.count += 1
                return
            elif slot == key:
                self.data[index] = value
                return
            i += 1
            if i >= self.size:
                raise RuntimeError("HashTable is full; rehash failed.")
            index = self.rehash(start, i, self.size)

    # 3) Задание 3: оператор in
    def __contains__(self, key):
        # поиск по ключу, не полагаясь на значение
        return self._find_key_index(key) is not None

    def _find_key_index(self, key):
        # базовый поиск без tombstone (будет переопределён ниже)
        start = self.hashfunction(key, self.size)
        index = start
        i = 0
        while True:
            slot = self.slots[index]
            if slot is None:
                return None
            if slot == key:
                return index
            i += 1
            if i >= self.size:
                return None
            index = self.rehash(start, i, self.size)

    # 4) Задание 4: авто-увеличение размера при LF > 0.7
    @staticmethod
    def _is_prime(n):
        # проверка простоты для подбора размера
        if n <= 1:
            return False
        if n <= 3:
            return True
        if n % 2 == 0 or n % 3 == 0:
            return False
        i = 5
        while i * i <= n:
            if n % i == 0 or n % (i + 2) == 0:
                return False
            i += 6
        return True

    @classmethod
    def _next_prime(cls, n):
        # следующее простое >= n
        if n <= 2:
            return 2
        p = n if n % 2 == 1 else n + 1
        while not cls._is_prime(p):
            p += 2
        return p

    def _resize(self, new_size):
        # изменение размера и реинсерция (убирает накопленные особенности пробирования)
        old_slots = self.slots
        old_data = self.data
        old_count = self.count

        self.size = self._next_prime(new_size)
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0

        for i in range(len(old_slots)):
            k = old_slots[i]
            if k is not None and k is not getattr(self, "_DELETED", None):
                self.put(k, old_data[i])

        assert self.count == old_count

    def __init__(self, initial_size=11):
        # инициализация с prime-округлением
        self.size = self._next_prime(max(11, initial_size))
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0

    def put(self, key, value):
        # вставка с авто-расширением при LF > 0.7
        if (self.count + 1) / self.size > 0.7:
            self._resize(self.size * 2)

        start = self.hashfunction(key, self.size)
        index = start
        i = 0
        while True:
            slot = self.slots[index]
            if slot is None:
                self.slots[index] = key
                self.data[index] = value
                self.count += 1
                return
            elif slot == key:
                self.data[index] = value
                return
            i += 1
            if i >= self.size:
                raise RuntimeError("HashTable is full; rehash failed.")
            index = self.rehash(start, i, self.size)

    # 5) Задание 5: удаление (del), tombstone и авто-уменьшение при LF < 0.2
    _DELETED = object()      # специальная метка удалённого элемента
    _MIN_SIZE = 11           # минимальный размер таблицы

    @classmethod
    def _prev_prime(cls, n):
        # предыдущее простое <= n
        if n <= 2:
            return 2
        p = n if n % 2 == 1 else n - 1
        while p > 2 and not cls._is_prime(p):
            p -= 2
        return p

    def __delitem__(self, key):
        # удаление с tombstone и авто-уменьшение
        idx = self._find_key_index(key)
        if idx is None:
            raise KeyError(key)
        self.slots[idx] = self._DELETED
        self.data[idx] = None
        self.count -= 1

        if self.size > self._MIN_SIZE and (self.count / self.size) < 0.2:
            target = max(self._MIN_SIZE, self.size // 2)
            new_size = max(self._prev_prime(target), self._MIN_SIZE)
            if new_size < self.size:
                self._resize(new_size)

    def _find_key_index(self, key):
        # поиск с учётом tombstone и квадратичного пробирования
        start = self.hashfunction(key, self.size)
        index = start
        i = 0
        while True:
            slot = self.slots[index]
            if slot is None:
                return None
            if slot is not self._DELETED and slot == key:
                return index
            i += 1
            if i >= self.size:
                return None
            index = self.rehash(start, i, self.size)

    def put(self, key, value):
        # вставка с переиспользованием tombstone и авто-ресайзами
        if (self.count + 1) / self.size > 0.7:
            self._resize(self.size * 2)

        start = self.hashfunction(key, self.size)
        index = start
        i = 0
        first_deleted = None

        while True:
            slot = self.slots[index]
            if slot is None:
                target_index = first_deleted if first_deleted is not None else index
                self.slots[target_index] = key
                self.data[target_index] = value
                self.count += 1
                return
            elif slot is self._DELETED:
                if first_deleted is None:
                    first_deleted = index
            elif slot == key:
                self.data[index] = value
                return

            i += 1
            if i >= self.size:
                raise RuntimeError("HashTable is full; rehash failed.")
            index = self.rehash(start, i, self.size)

    def _resize(self, new_size):
        # реинсерция убирает tombstone
        old_slots = self.slots
        old_data = self.data
        old_count = self.count

        self.size = self._next_prime(new_size)
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0

        for i in range(len(old_slots)):
            k = old_slots[i]
            if k is not None and k is not self._DELETED:
                self.put(k, old_data[i])

        assert self.count == old_count


def test_quadratic_probing_and_basic_put_get():
    # 1) Задание 1: квадратичное пробирование и корректность put/get при коллизиях
    H = HashTable()
    keys = [0, 11, 22, 33, 44]  # все дают один и тот же стартовый слот при size=11
    vals = [f"v{k}" for k in keys]
    for k, v in zip(keys, vals):
        H[k] = v
    for k, v in zip(keys, vals):
        assert H[k] == v

    H[22] = "updated"
    assert H[22] == "updated"

def test_len_and_contains():
    # 2) Задание 2 и 3: поддержка len() и оператора in
    H = HashTable()
    assert len(H) == 0
    assert (10 in H) is False

    H[10] = "a"
    H[21] = "b"
    assert len(H) == 2
    assert (10 in H) is True
    assert (21 in H) is True
    assert (32 in H) is False

def test_auto_resize_up_preserves_data():
    # 3) Задание 4: авто-расширение при LF > 0.7 и сохранность данных
    H = HashTable(initial_size=11)
    for i in range(8):  # 8/11 > 0.7 => должен быть resize
        H[i] = f"v{i}"
    assert H.size >= 23
    for i in range(8):
        assert H[i] == f"v{i}"

def test_delete_and_auto_shrink():
    # 4) Задание 5: удаление и авто-уменьшение при LF < 0.2
    #    (также проверяется авто-расширение из Задания 4 при первичном наполнении)
    H = HashTable(initial_size=11)
    for i in range(12):
        H[i] = f"v{i}"
    old_size = H.size
    assert old_size >= 23

    for i in range(8):
        del H[i]
    assert len(H) == 4
    assert H.size <= old_size and H.size >= 11
    for i in range(8, 12):
        assert H[i] == f"v{i}"

def test_tombstone_preserves_chain_and_reuse():
    # 5) Задание 5: tombstone сохраняет цепочку поиска и переиспользуется при вставке
    H = HashTable(initial_size=11)
    keys = [0, 11, 22, 33]
    for k in keys:
        H[k] = f"v{k}"
    del H[11]
    assert (11 in H) is False
    assert H[22] == "v22"
    assert H[33] == "v33"
    H[44] = "v44"  # должен использовать tombstone
    assert H[44] == "v44"

def test_delete_missing_raises():
    # 6) Дополнительно к Заданию 5: del по отсутствующему ключу должен поднимать KeyError
    H = HashTable()
    try:
        del H[123]
        assert False, "Expected KeyError"
    except KeyError:
        pass

def test_len_stable_on_update():
    # 7) Дополнительно к Заданию 2: len не меняется при обновлении существующего ключа
    H = HashTable()
    H[5] = "a"
    n1 = len(H)
    H[5] = "b"
    assert len(H) == n1
    assert H[5] == "b"

def run_all():
    print("1) Задание 1: квадратичное пробирование")
    test_quadratic_probing_and_basic_put_get()

    print("2) Задание 2 и 3: len() и оператор in")
    test_len_and_contains()

    print("3) Задание 4: авто-расширение при LF > 0.7")
    test_auto_resize_up_preserves_data()

    print("4) Задание 5: удаление и авто-уменьшение при LF < 0.2")
    test_delete_and_auto_shrink()

    print("5) Задание 5: tombstone — сохранение цепочки и переиспользование")
    test_tombstone_preserves_chain_and_reuse()

    print("6) Дополнительно к Заданию 5: KeyError при удалении отсутствующего ключа")
    test_delete_missing_raises()

    print("7) Дополнительно к Заданию 2: стабильность len при обновлении")
    test_len_stable_on_update()

    print("All tests passed.")

if __name__ == "__main__":
    run_all()

1) Задание 1: квадратичное пробирование
2) Задание 2 и 3: len() и оператор in
3) Задание 4: авто-расширение при LF > 0.7
4) Задание 5: удаление и авто-уменьшение при LF < 0.2
5) Задание 5: tombstone — сохранение цепочки и переиспользование
6) Дополнительно к Заданию 5: KeyError при удалении отсутствующего ключа
7) Дополнительно к Заданию 2: стабильность len при обновлении
All tests passed.


### № 2
Возьмите реализацию класса HashTable из лекционных материалов и выполните
следующие доработки:
1. Переделайте существующие методы так, чтобы разрешение коллизий происходило
не при помощи концепции открытой адресации, а методом цепочек. Для этого в
каждом слоте храните связный список, реализованный классом UnorderedList из
лабораторной работы 3.
2. Реализуйте работу с функцией len (переопределите метод __len__).
3. Реализуйте работу оператора in (переопределите метод __contains__).
4. Переделайте метод put таким образом, чтобы таблица автоматически меняла размер,
когда загрузочный фактор становится больше значения 0.7. Размер должен
увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте работу оператора del (переопределите метод __delitem__) для удаления
элемента таблицы. Таблица должна автоматически менять размер, когда
загрузочный фактор становится меньше значения 0.2. Размер должен уменьшаться
примерно в два раза до ближайшего подходящего простого числа.
Все выполненные доработки должны быть протестированы.

In [10]:
class Node:
    def __init__(self, initdata):
        self.data = initdata
        self.next = None

    def getData(self):
        return self.data

    def getNext(self):
        return self.next

    def setData(self, newdata):
        self.data = newdata

    def setNext(self, newnext):
        self.next = newnext


class UnorderedList:

    def __init__(self):
        self.head = None

    def isEmpty(self):
        return self.head is None

    def add(self, item):
        temp = Node(item)
        temp.setNext(self.head)
        self.head = temp

    def size(self):
        current = self.head
        count = 0
        while current is not None:
            count += 1
            current = current.getNext()
        return count

    def search(self, item):
        current = self.head
        found = False
        while current is not None and not found:
            if current.getData() == item:
                found = True
            else:
                current = current.getNext()
        return found

    def remove(self, item):
        current = self.head
        previous = None
        found = False
        while not found:
            if current.getData() == item:
                found = True
            else:
                previous = current
                current = current.getNext()
            if current is None:
                raise ValueError("Item not found in list")
        if previous is None:
            self.head = current.getNext()
        else:
            previous.setNext(current.getNext())

    # Доработки ниже

    def append(self, item):
        """Добавляет элемент в конец списка"""
        temp = Node(item)
        if self.head is None:
            self.head = temp
        else:
            current = self.head
            while current.getNext() is not None:
                current = current.getNext()
            current.setNext(temp)

    def index(self, item):
        """Возвращает индекс первого вхождения item, иначе ValueError"""
        current = self.head
        idx = 0
        while current is not None:
            if current.getData() == item:
                return idx
            current = current.getNext()
            idx += 1
        raise ValueError("Item not found in list")

    def insert(self, pos, item):
        """Вставляет элемент item на позицию pos"""
        if pos < 0 or pos > self.size():
            raise IndexError("Index out of range")
        temp = Node(item)
        if pos == 0:
            temp.setNext(self.head)
            self.head = temp
        else:
            previous = None
            current = self.head
            idx = 0
            while idx < pos:
                previous = current
                current = current.getNext()
                idx += 1
            previous.setNext(temp)
            temp.setNext(current)

    def pop(self, pos=None):
        """Удаляет и возвращает элемент по индексу pos, либо последний, если pos не указан"""
        sz = self.size()
        if sz == 0:
            raise IndexError("pop from empty list")
        if pos is None:
            pos = sz - 1
        if pos < 0 or pos >= sz:
            raise IndexError("pop index out of range")
        previous = None
        current = self.head
        idx = 0
        while idx < pos:
            previous = current
            current = current.getNext()
            idx += 1
        if previous is None:
            self.head = current.getNext()
        else:
            previous.setNext(current.getNext())
        return current.getData()

    def __str__(self):
        """Строковое представление списка как [a, b, c]"""
        result = []
        current = self.head
        while current is not None:
            result.append(repr(current.getData()))
            current = current.getNext()
        return '[' + ', '.join(result) + ']'

    def slice(self, start, stop):
        """Возвращает копию списка, начиная с позиции start и заканчивая (не включая) stop"""
        if start < 0 or stop < 0 or start > self.size() or stop > self.size() or start > stop:
            raise IndexError("Invalid slice indices")
        new_list = UnorderedList()
        current = self.head
        idx = 0
        while current is not None and idx < stop:
            if idx >= start:
                new_list.append(current.getData())
            current = current.getNext()
            idx += 1
        return new_list

In [11]:
class HashTable:
    def __init__(self):
        self.size = 11
        self.slots = [None] * self.size
        self.data = [None] * self.size

    def put(self, key, data):
        hashvalue = self.hashfunction(key, len(self.slots))

        if self.slots[hashvalue] == None:
            self.slots[hashvalue] = key
            self.data[hashvalue] = data
        else:
            if self.slots[hashvalue] == key:
                self.data[hashvalue] = data  # replace
            else:
                nextslot = self.rehash(hashvalue, len(self.slots))
                while self.slots[nextslot] != None and \
                        self.slots[nextslot] != key:
                    nextslot = self.rehash(nextslot, len(self.slots))

                if self.slots[nextslot] == None:
                    self.slots[nextslot] = key
                    self.data[nextslot] = data
                else:
                    self.data[nextslot] = data

    def hashfunction(self, key, size):
        return key % size

    def rehash(self, oldhash, size):
        return (oldhash + 1) % size

    def get(self, key):
        startslot = self.hashfunction(key, len(self.slots))

        data = None
        stop = False
        found = False
        position = startslot
        while self.slots[position] != None and \
                not found and not stop:
            if self.slots[position] == key:
                found = True
                data = self.data[position]
            else:
                position = self.rehash(position, len(self.slots))
                if position == startslot:
                    stop = True
        return data

    def __getitem__(self, key):
        return self.get(key)

    def __setitem__(self, key, data):
        self.put(key, data)

    # 1) Задание 1: переход на метод цепочек (в каждом слоте связный список UnorderedList)
    def __init__(self, initial_size=11):
        # инициализация бакетов-списков вместо параллельных массивов
        self.size = self._next_prime(max(11, initial_size))
        self.buckets = [UnorderedList() for _ in range(self.size)]
        self.count = 0  # количество пар (ключ, значение)

    def put(self, key, value):
        # вставка/обновление в бакет (цепочки); авто-рост при LF > 0.7 — см. Задание 4
        if (self.count + 1) / self.size > 0.7:  # Задание 4: порог расширения
            self._resize(self._next_prime(self.size * 2))

        idx = self.hashfunction(key, self.size)
        bucket = self.buckets[idx]

        prev, node = self._bucket_find_pair(bucket, key)  # поиск по ключу в списке
        if node is not None:
            k, _ = node.getData()
            node.setData((k, value))  # обновление по ключу
        else:
            bucket.add((key, value))  # добавляем в голову списка
            self.count += 1

    def get(self, key):
        # поиск значения по ключу в соответствующем бакете-списке
        idx = self.hashfunction(key, self.size)
        bucket = self.buckets[idx]
        _, node = self._bucket_find_pair(bucket, key)
        if node is None:
            return None
        return node.getData()[1]

    # 2) Задание 2: поддержка len()
    def __len__(self):
        return self.count

    # 3) Задание 3: оператор in
    def __contains__(self, key):
        idx = self.hashfunction(key, self.size)
        _, node = self._bucket_find_pair(self.buckets[idx], key)
        return node is not None

    # 5) Задание 5: оператор del с авто-уменьшением при LF < 0.2
    def __delitem__(self, key):
        idx = self.hashfunction(key, self.size)
        bucket = self.buckets[idx]

        # удаление узла с данным ключом из связного списка
        prev, node = self._bucket_find_pair(bucket, key)
        if node is None:
            raise KeyError(key)

        if prev is None:
            bucket.head = node.getNext()
        else:
            prev.setNext(node.getNext())
        self.count -= 1

        # авто-уменьшение размера
        if self.size > self._MIN_SIZE and (self.count / self.size) < 0.2:
            target = max(self._MIN_SIZE, self.size // 2)
            new_size = max(self._prev_prime(target), self._MIN_SIZE)
            if new_size < self.size:
                self._resize(new_size)

    # 1) Задание 1: служебный поиск пары (prev, node) в бакете по ключу
    def _bucket_find_pair(self, bucket: UnorderedList, key):
        prev = None
        current = bucket.head
        while current is not None:
            k, v = current.getData()
            if k == key:
                return prev, current
            prev = current
            current = current.getNext()
        return None, None

    # 4/5) Задание 4 и 5: примитивы простых чисел и изменение размера
    @staticmethod
    def _is_prime(n):
        if n <= 1:
            return False
        if n <= 3:
            return True
        if n % 2 == 0 or n % 3 == 0:
            return False
        i = 5
        while i * i <= n:
            if n % i == 0 or n % (i + 2) == 0:
                return False
            i += 6
        return True

    @classmethod
    def _next_prime(cls, n):
        if n <= 2:
            return 2
        p = n if n % 2 == 1 else n + 1
        while not cls._is_prime(p):
            p += 2
        return p

    @classmethod
    def _prev_prime(cls, n):
        if n <= 2:
            return 2
        p = n if n % 2 == 1 else n - 1
        while p > 2 and not cls._is_prime(p):
            p -= 2
        return p

    _MIN_SIZE = 11  # 5) Задание 5: минимальный размер таблицы

    def _resize(self, new_size):
        # перенос всех пар в новую таблицу (перехеширование)
        old_buckets = self.buckets
        old_count = self.count

        self.size = self._next_prime(new_size)
        self.buckets = [UnorderedList() for _ in range(self.size)]
        self.count = 0

        for bucket in old_buckets:
            current = bucket.head
            while current is not None:
                k, v = current.getData()
                # вставка без повторного ресайза
                idx = self.hashfunction(k, self.size)
                self.buckets[idx].add((k, v))
                self.count += 1
                current = current.getNext()

        assert self.count == old_count

# Тесты с подписями по заданиям (запускать по ячейкам в Jupyter)

def test_chaining_basic_put_get_and_update():
    # 1) Задание 1: метод цепочек — корректная работа при коллизиях и обновлении
    H = HashTable(initial_size=11)
    keys = [0, 11, 22, 33, 44]  # все попадают в один бакет (индекс 0)
    vals = [f"v{k}" for k in keys]
    for k, v in zip(keys, vals):
        H[k] = v
    for k, v in zip(keys, vals):
        assert H[k] == v

    H[22] = "updated"
    assert H[22] == "updated"

def test_len_and_contains():
    # 2) Задание 2 и 3: len() и оператор in
    H = HashTable()
    assert len(H) == 0
    assert (10 in H) is False

    H[10] = "a"
    H[21] = "b"   # может попасть в тот же бакет или другой — неважно
    assert len(H) == 2
    assert (10 in H) is True
    assert (21 in H) is True
    assert (32 in H) is False

def test_auto_resize_up_preserves_data():
    # 4) Задание 4: авто-расширение при LF > 0.7 и сохранность данных
    H = HashTable(initial_size=11)
    for i in range(8):  # 8/11 > 0.7 => должен быть resize
        H[i] = f"v{i}"
    assert H.size >= 23
    for i in range(8):
        assert H[i] == f"v{i}"

def test_delete_and_auto_shrink():
    # 5) Задание 5: удаление с цепочками и авто-уменьшение при LF < 0.2
    H = HashTable(initial_size=11)
    for i in range(12):  # гарантируем рост
        H[i] = f"v{i}"
    old_size = H.size
    assert old_size >= 23

    for i in range(8):  # удалим 8 — останется 4 элемента
        del H[i]
    assert len(H) == 4
    assert H.size <= old_size and H.size >= 11
    for i in range(8, 12):
        assert H[i] == f"v{i}"

def test_delete_missing_raises():
    # 5) Задание 5: del по отсутствующему ключу поднимает KeyError
    H = HashTable()
    try:
        del H[123]
        assert False, "Expected KeyError"
    except KeyError:
        pass

def test_len_stable_on_update():
    # 2) Задание 2: len не меняется при обновлении существующего ключа
    H = HashTable()
    H[5] = "a"
    n1 = len(H)
    H[5] = "b"
    assert len(H) == n1
    assert H[5] == "b"

def test_chain_integrity_after_delete():
    # 1) Задание 1 + 5: удаление в бакете не ломает остальные элементы
    H = HashTable(initial_size=11)
    keys = [0, 11, 22, 33]
    for k in keys:
        H[k] = f"v{k}"
    del H[11]
    assert (11 in H) is False
    assert H[22] == "v22"
    assert H[33] == "v33"
    H[44] = "v44"  # просто новая вставка в тот же бакет
    assert H[44] == "v44"

def run_all():
    print("1) Задание 1: метод цепочек — put/get/обновление/коллизии")
    test_chaining_basic_put_get_and_update()

    print("2) Задание 2 и 3: len() и оператор in")
    test_len_and_contains()

    print("3) Задание 4: авто-расширение при LF > 0.7")
    test_auto_resize_up_preserves_data()

    print("4) Задание 5: удаление и авто-уменьшение при LF < 0.2")
    test_delete_and_auto_shrink()

    print("5) Задание 5: удаление в цепочке не ломает остальные элементы")
    test_chain_integrity_after_delete()

    print("6) Доп. к Заданию 5: KeyError при удалении отсутствующего ключа")
    test_delete_missing_raises()

    print("7) Доп. к Заданию 2: стабильность len при обновлении")
    test_len_stable_on_update()

    print("All tests passed")

# Запуск
run_all()

1) Задание 1: метод цепочек — put/get/обновление/коллизии
2) Задание 2 и 3: len() и оператор in
3) Задание 4: авто-расширение при LF > 0.7
4) Задание 5: удаление и авто-уменьшение при LF < 0.2
5) Задание 5: удаление в цепочке не ломает остальные элементы
6) Доп. к Заданию 5: KeyError при удалении отсутствующего ключа
7) Доп. к Заданию 2: стабильность len при обновлении
All tests passed.


### № 3
Переделайте класс HashTable, чтобы в качестве ключей можно было использовать строки.

In [13]:
def hashfunction_str_int(self, key, size):
    # Поддержка строковых ключей (и сохранение работы с int)
    if isinstance(key, int):
        return key % size
    if isinstance(key, str):
        h = 0
        for ch in key:
            h = h * 31 + ord(ch)
        return h % size
    raise TypeError("Unsupported key type. Use int or str.")

HashTable.hashfunction = hashfunction_str_int

def run_string_key_tests():
    print("Тест 1: строки — put/get/обновление, len, in")
    H = HashTable(initial_size=11)
    H["cat"] = 1
    H["dog"] = 2
    assert len(H) == 2
    assert ("cat" in H) and ("dog" in H) and ("lion" not in H)
    assert H["cat"] == 1 and H["dog"] == 2
    H["dog"] = 20  # обновление
    assert H["dog"] == 20

    print("Тест 2: авто-расширение при LF > 0.7")
    H = HashTable(initial_size=11)
    for i in range(8):  # 8/11 > 0.7
        H[f"k{i}"] = i
    assert H.size >= 23  # увеличилась до ближайшего простого ~2x
    for i in range(8):
        assert H[f"k{i}"] == i  # данные сохранились

    print("Тест 3: удаление и авто-уменьшение при LF < 0.2")
    H = HashTable(initial_size=11)
    for i in range(12):  # спровоцируем рост
        H[f"w{i}"] = i
    old_size = H.size
    for i in range(8):
        del H[f"w{i}"]
    assert len(H) == 4
    assert H.size <= old_size and H.size >= 11  # уменьшилась, но не ниже минимума
    for i in range(8, 12):
        assert H[f"w{i}"] == i  # оставшиеся доступны

    print("Тест 4: смешанные ключи (str + int)")
    H = HashTable(initial_size=11)
    H["x"] = 100
    H[42] = "answer"
    assert H["x"] == 100 and H[42] == "answer"

    print("All tests passed")

run_string_key_tests()

Тест 1: строки — put/get/обновление, len, in
Тест 2: авто-расширение при LF > 0.7
Тест 3: удаление и авто-уменьшение при LF < 0.2
Тест 4: смешанные ключи (str + int)
All tests passed


### № 4
Дана строчка русского текста, состоящая из слов и пробелов. Словом считается
последовательность русских букв, слова разделены одним или большим числом пробелов.
Для каждого слова этого текста узнайте порядковый номер его вхождения в текст именно в
той форме, в которой указано слово. Для первого вхождения слова выведите «1», для
второго вхождения того же слова выведите «2» и так далее.
Для решения этой задачи используйте класс HashTable из задания № 3.

In [15]:
text = "Раз раз раз как меня слышно Повторяю раз раз раз Повторяю"
words = text.split()
H = HashTable(initial_size=11)
orders = []

for w in words:
    if w in H:
        cnt = H[w] + 1
        H[w] = cnt
        orders.append(str(cnt))
    else:
        H[w] = 1
        orders.append("1")

result = ' '.join(orders)
print(result)

# Быстрая проверка ожидаемого ответа:
expected = "1 1 2 1 1 1 1 3 4 5 2"
assert result == expected

1 1 2 1 1 1 1 3 4 5 2


### № 5
Напишите программу, имитирующую процесс регистрации и авторизации. Для каждого
пользователя программа должна сохранять логин, хеш его пароля и «соль». Для хранения
данных можно использовать БД или файл.
Действия при сохранении пароля:
1. Сгенерируйте длинную случайную «соль» при помощи модуля secrets (secrets —
Generate secure random numbers for managing secrets — Python 3.10.0 documentation).
Длина «соли» должна быть такой же как и выходные данные используемой вами
хэш-функции. Например, если для хеширования вы используете SHA256, то на
выходе вы получите 256 бит (32 байта). В этом случае соль должна составлять не
менее 32 случайных байт.
2. Добавьте «соль» к паролю и хэшируйте его с помощью функции scrypt из модуля
hashlib (Хеширование паролей модулем hashlib в Python. (docs-python.ru)).
3. Сохраните логин, «соль» и получившейся хэш в БД или в файл.
Действия при проверке пароля:
1. Извлеките «соль» и хэш пользователя из БД или файла.
2. Добавьте «соль» к введенному паролю и хэшируйте его, используя ту же хэшфункцию, что и в алгоритме сохранения пароля.
3. Сравните получившейся хэш введенного пароля с хэшом из БД или файла. Если они
совпадают, то пароль правильный. В противном случае пароль был введен неверно.

In [19]:
import json
import base64
import secrets

DB_PATH = "users_demo.json"

# Параметры KDF scrypt
SCRYPT_PARAMS = {
    "n": 2**14,  # 16384
    "r": 8,
    "p": 1,
    "dklen": 32
}

# Загрузка БД
def _load_db(path=DB_PATH):
    if not os.path.exists(path):
        return {
            "version": 1,
            "kdf": "scrypt",
            "params": SCRYPT_PARAMS,
            "users": {}
        }
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# Сохранение БД в файл
def _save_db(db, path=DB_PATH):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(db, f, ensure_ascii=False, indent=2)

# Хеширование пароля scrypt с заданной солью и параметрами
def _scrypt_hash(password: str, salt: bytes, params: dict) -> bytes:
    return hashlib.scrypt(
        password=password.encode("utf-8"),
        salt=salt,
        n=params["n"],
        r=params["r"],
        p=params["p"],
        dklen=params["dklen"]
    )

# Регистрация пользователя (логин, пароль) — генерируем соль и сохраняем хеш
def register_user(username: str, password: str, path=DB_PATH):
    db = _load_db(path)
    users = db["users"]
    if username in users:
        raise ValueError("User already exists")

    params = db["params"]
    salt_len = params["dklen"]  # длина соли как длина выходных данных
    salt = secrets.token_bytes(salt_len)
    pwd_hash = _scrypt_hash(password, salt, params)

    # храним соль и хеш в base64-строках
    users[username] = {
        "salt": base64.b64encode(salt).decode("ascii"),
        "hash": base64.b64encode(pwd_hash).decode("ascii")
    }
    _save_db(db, path)

# Проверка пароля — извлекаем соль/хеш, считаем scrypt и сравниваем
def verify_user(username: str, password: str, path=DB_PATH) -> bool:
    db = _load_db(path)
    users = db["users"]
    rec = users.get(username)
    if rec is None:
        return False

    params = db["params"]
    salt = base64.b64decode(rec["salt"])
    stored_hash = base64.b64decode(rec["hash"])
    cand_hash = _scrypt_hash(password, salt, params)

    # защищённое сравнение
    return secrets.compare_digest(stored_hash, cand_hash)


while True:
    print("\nВыберите действие:")
    print("  1) Регистрация")
    print("  2) Вход")
    print("  3) Выход")
    choice = input("Ваш выбор [1-3]: ").strip()

    if choice == "1":
        username = input("Логин: ").strip()
        if not username:
            print("Логин не может быть пустым.")
            continue
        pwd1 = input("Пароль: ")
        pwd2 = input("Повторите пароль: ")
        if pwd1 != pwd2:
            print("Пароли не совпадают.")
            continue
        try:
            register_user(username, pwd1)
            print("Регистрация успешна.")
        except ValueError as e:
            print(f"Ошибка: {e}")
        except Exception as e:
            print(f"Не удалось зарегистрировать: {e}")

    elif choice == "2":
        username = input("Логин: ").strip()
        if not username:
            print("Логин не может быть пустым.")
            continue
        pwd = input("Пароль: ")
        try:
            if verify_user(username, pwd):
                print("Вход выполнен успешно.")
            else:
                print("Неверный логин или пароль.")
        except Exception as e:
            print(f"Не удалось выполнить вход: {e}")

    elif choice == "3":
        print("Выход.")
        break

    else:
        print("Неизвестная команда. Введите 1, 2 или 3.")


Выберите действие:
  1) Регистрация
  2) Вход
  3) Выход
Ошибка: User already exists

Выберите действие:
  1) Регистрация
  2) Вход
  3) Выход
Неверный логин или пароль.

Выберите действие:
  1) Регистрация
  2) Вход
  3) Выход
Регистрация успешна.

Выберите действие:
  1) Регистрация
  2) Вход
  3) Выход
Неизвестная команда. Введите 1, 2 или 3.

Выберите действие:
  1) Регистрация
  2) Вход
  3) Выход
Неизвестная команда. Введите 1, 2 или 3.

Выберите действие:
  1) Регистрация
  2) Вход
  3) Выход
Выход.


### № 6
Напишите программу, которая принимает от пользователя путь до директории. Для всех
файлов из данной директории должен быть вычислен хеш. Программа должна выявить и
вывести на экран все дубликаты в этой директории (т.е. файлы, у которых одинаковый хеш).

In [23]:
import os
import hashlib

def file_hash(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print("Поиск дубликатов файлов по хешу (SHA-256)")
root = input("Введите путь до директории: ").strip()

if not os.path.isdir(root):
    print("Ошибка: указан несуществующий путь или это не директория.")
else:
    hash_to_paths = {}
    total_files = 0
    errors = 0

    try:
        with os.scandir(root) as it:
            for entry in it:
                if entry.is_file(follow_symlinks=False):
                    try:
                        total_files += 1
                        h = file_hash(entry.path)
                        hash_to_paths.setdefault(h, []).append(entry.path)
                    except Exception:
                        errors += 1
    except Exception as e:
        print(f"Ошибка при сканировании: {e}")

    duplicates = {h: paths for h, paths in hash_to_paths.items() if len(paths) > 1}

    print(f"\nПросканировано файлов: {total_files}")
    if errors:
        print(f"Предупреждение: файлов с ошибками чтения/хеширования: {errors}")
    if not duplicates:
        print("Дубликатов не найдено.")
    else:
        print(f"Найдено групп дубликатов: {len(duplicates)}")
        for i, (h, paths) in enumerate(duplicates.items(), start=1):
            print(f"\nГруппа {i}: хеш {h} — {len(paths)} файла(ов)")
            for p in paths:
                print(f"  - {p}")

Поиск дубликатов файлов по хешу (SHA-256)

Просканировано файлов: 5
Найдено групп дубликатов: 1

Группа 1: хеш ccf2c058d9411e946d2aa6daf53588e48d5654a480c11d3aca2f0591025287df — 3 файла(ов)
  - C:\Users\danil\OneDrive\Desktop\study\metodi\otchet — копия (2).docx
  - C:\Users\danil\OneDrive\Desktop\study\metodi\otchet — копия.docx
  - C:\Users\danil\OneDrive\Desktop\study\metodi\otchet.docx
